# StanChart/ASM Two-Stage PCA Scanner — USD SOFR Swap Curve

**Strategy:** Two-stage PCA trade scanner. Stage 1 on full curve for Z-score surface, Stage 2 on 3 trade tenors for PCA-orthogonal weights.

**References:**
- Standard Chartered — "Introducing a relative-value tool for swaps" (Lee, Davies, Fernandez — Aug 2013)
- ASM Quant Macro — "PCA Part II: Constructing Butterflies" (bquanttrading 2015)
- Salomon Smith Barney — "Principles of Principal Components" (Ilmanen, Iwanowski — Jan 2000)

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapValue import IRSwapValue

from BT.signals.pca_rv_engine import (
    PCARVConfig, rolling_pca, pca_fly_weights, ou_params, adf_test,
)
from BT.signals.rv_backtest import RVBacktestConfig, run_rv_backtest

In [ ]:
config = {
    # === DATA ===
    "tenors_spot": ["1Y","2Y","3Y","4Y","5Y","7Y","8Y","9Y","10Y","15Y","20Y","25Y","30Y"],
    "forward_starts": ["1M","3M","6M","1Y","2Y","5Y","7Y","10Y"],
    "data_start": "2021-01-01",
    "data_end": None,  # latest available
    "pca_window_days": 520,  # 2Y default (StanChart)
    "mtm_mode": "approximate",  # "approximate" or "curve"
    # === STAGE 1: FULL-CURVE PCA ===
    "stage1_n_components": 3,
    "stage1_pca_input": "levels",
    "stage1_zscore_lookback_days": 130,
    "stage1_zscore_lookback_options": [65, 130, 260, 520],
    # === STAGE 2: TRADE-SPECIFIC PCA ===
    "stage2_n_components": 3,
    "stage2_pca_input": "levels",
    # === TRADE IDENTIFICATION ===
    "butterfly_scan": {
        "min_belly_zscore": 1.5,
        "min_wing_zscore": 0.5,
        "require_opposite_sign": True,
    },
    # === MEAN REVERSION ===
    "ou_estimation_window_days": 520,
    "min_half_life_days": 3,
    "max_half_life_days": 120,
    "adf_significance": 0.05,
    # === CARRY ===
    "carry_horizon": "1M",
    "include_carry_in_ranking": True,
    # === COSTS ===
    "round_trip_cost_bp": 0.5,
    "min_profit_to_cost_ratio": 2.0,
    # === BACKTEST (lightweight) ===
    "backtest_enabled": True,
    "backtest_entry_zscore": 1.5,
    "backtest_exit_mean_reversion": True,
    "backtest_exit_max_days": 22,
    "backtest_exit_stop_sd": 2.0,
}

In [ ]:
curve_mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-RL_BASIC")
irswaps_tb = IRSwapsTB(curve_mdp, show_tqdm=True)
tb = TimeseriesBuilder(irswaps_tb=irswaps_tb)

end_date = config["data_end"] or datetime.date.today()
if isinstance(end_date, str):
    end_date = datetime.datetime.strptime(end_date, "%Y-%m-%d").date()
start_date = datetime.datetime.strptime(config["data_start"], "%Y-%m-%d").date()

# Build queries for each forward start × spot tenor
queries = []
tenor_labels = {}  # query -> human-readable label

for fwd in ["Spot"] + config["forward_starts"]:
    for tenor in config["tenors_spot"]:
        if fwd == "Spot":
            q = IRSwapQuery(curve="USD-SOFR-1D", tenor=tenor, value=IRSwapValue.RATE)
            label = f"spot_{tenor}"
        else:
            fwd_tenor = f"{fwd}{tenor}"  # e.g., "1Y5Y" = 1Y forward 5Y swap
            q = IRSwapQuery(curve="USD-SOFR-1D", tenor=fwd_tenor, value=IRSwapValue.RATE)
            label = f"{fwd}x{tenor}"
        queries.append(q)
        tenor_labels[q] = label

rates_df = tb.get_timeseries(
    start=start_date, end=end_date, queries=queries, n_jobs=8,
)

# Rename columns to human-readable labels
col_rename = {}
for q, label in tenor_labels.items():
    col = q.col_name(cube_name="USD-SOFR-1D")
    if col in rates_df.columns:
        col_rename[col] = label
rates_df = rates_df.rename(columns=col_rename)

print(f"Loaded: {rates_df.shape[0]} dates x {rates_df.shape[1]} tenors")
rates_df.tail(3)

In [ ]:
pca_config = PCARVConfig(
    pca_window_days=config["pca_window_days"],
    pca_input=config["stage1_pca_input"],
    n_components=config["stage1_n_components"],
    zscore_lookback_days=config["stage1_zscore_lookback_days"],
)

# Group columns by forward start
curve_groups = {}
for col in rates_df.columns:
    if col.startswith("spot_"):
        curve_groups.setdefault("Spot", []).append(col)
    else:
        fwd = col.split("x")[0]
        curve_groups.setdefault(fwd, []).append(col)

# Run PCA per curve
pca_results = {}
for curve_name, cols in sorted(curve_groups.items()):
    df_curve = rates_df[cols].dropna(axis=0, how="any")
    if len(df_curve) < pca_config.pca_window_days + 50:
        print(f"Skipping {curve_name}: insufficient data ({len(df_curve)} rows)")
        continue
    pca_results[curve_name] = rolling_pca(df_curve, pca_config)
    print(f"{curve_name}: {len(df_curve)} dates, "
          f"VE = {pca_results[curve_name].variance_explained.dropna().iloc[-1].values}")

## Z-Score Surface & Butterfly Scanner

In [ ]:
# Build Z-score surface: rows = tenors, columns = forward starts
latest_date = rates_df.index[-1]
zscore_surface = pd.DataFrame(index=config["tenors_spot"])

for curve_name, result in pca_results.items():
    zs = result.zscores
    if latest_date in zs.index:
        for col in zs.columns:
            tenor = col.split("_")[-1] if "spot_" in col else col.split("x")[-1]
            zscore_surface.loc[tenor, curve_name] = zs.loc[latest_date, col]

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 8))
cmap = sns.diverging_palette(240, 10, as_cmap=True)
sns.heatmap(
    zscore_surface.astype(float), cmap=cmap, center=0,
    annot=True, fmt=".1f", linewidths=0.5, ax=ax,
    vmin=-3, vmax=3,
)
ax.set_title(f"PCA Residual Z-Scores — USD SOFR ({latest_date.strftime('%Y-%m-%d')})\n"
             f"+ = Cheap (yield too high), − = Rich (yield too low)")
ax.set_ylabel("Swap Tenor")
ax.set_xlabel("Forward Start")
plt.tight_layout()
plt.show()

In [ ]:
def scan_butterflies(zscore_surface, config):
    """Scan Z-score surface for butterfly candidates."""
    candidates = []
    min_belly = config["butterfly_scan"]["min_belly_zscore"]
    min_wing = config["butterfly_scan"]["min_wing_zscore"]
    require_opp = config["butterfly_scan"]["require_opposite_sign"]

    tenors = list(zscore_surface.index)
    for curve_name in zscore_surface.columns:
        zs = zscore_surface[curve_name].dropna()
        avail_tenors = list(zs.index)
        for i, belly_t in enumerate(avail_tenors):
            belly_z = zs[belly_t]
            if abs(belly_z) < min_belly:
                continue
            for left_t in avail_tenors[:i]:
                left_z = zs[left_t]
                if abs(left_z) < min_wing:
                    continue
                for right_t in avail_tenors[i+1:]:
                    right_z = zs[right_t]
                    if abs(right_z) < min_wing:
                        continue
                    if require_opp:
                        if not (np.sign(belly_z) != np.sign(left_z) and np.sign(belly_z) != np.sign(right_z)):
                            continue
                    candidates.append({
                        "curve": curve_name,
                        "left": left_t, "belly": belly_t, "right": right_t,
                        "z_left": left_z, "z_belly": belly_z, "z_right": right_z,
                        "z_total": abs(belly_z) + abs(left_z) + abs(right_z),
                        "direction": "receive_belly" if belly_z > 0 else "pay_belly",
                    })
    return pd.DataFrame(candidates).sort_values("z_total", ascending=False)

fly_candidates = scan_butterflies(zscore_surface, config)
print(f"Found {len(fly_candidates)} butterfly candidates")
display(fly_candidates.head(20))

In [ ]:
def compute_stage2_weights(rates_df, candidate_row, config):
    """Run Stage 2 PCA on just the 3 trade tenors for PCA-orthogonal weights."""
    curve = candidate_row["curve"]
    left_col = f"{curve}x{candidate_row['left']}" if curve != "Spot" else f"spot_{candidate_row['left']}"
    belly_col = f"{curve}x{candidate_row['belly']}" if curve != "Spot" else f"spot_{candidate_row['belly']}"
    right_col = f"{curve}x{candidate_row['right']}" if curve != "Spot" else f"spot_{candidate_row['right']}"

    cols = [left_col, belly_col, right_col]
    if not all(c in rates_df.columns for c in cols):
        return None

    df3 = rates_df[cols].dropna()
    pca_cfg = PCARVConfig(
        pca_window_days=config["pca_window_days"],
        pca_input=config["stage2_pca_input"],
        n_components=3,
        zscore_lookback_days=config["stage1_zscore_lookback_days"],
    )
    weights = pca_fly_weights(df3, pca_cfg)
    return weights

# Compute for top N candidates
top_n = min(10, len(fly_candidates))
for idx in fly_candidates.head(top_n).index:
    row = fly_candidates.loc[idx]
    w = compute_stage2_weights(rates_df, row, config)
    if w is not None and not w.dropna().empty:
        latest_w = w.dropna().iloc[-1]
        fly_candidates.loc[idx, "w_left"] = latest_w.iloc[0]
        fly_candidates.loc[idx, "w_belly"] = latest_w.iloc[1]
        fly_candidates.loc[idx, "w_right"] = latest_w.iloc[2]

display(fly_candidates.head(top_n)[["curve", "left", "belly", "right",
    "z_belly", "direction", "w_left", "w_belly", "w_right"]])

## OU Estimation & Trade Analytics

In [ ]:
def compute_trade_analytics(rates_df, candidate_row, weights_df, config):
    """Compute OU params, carry, and trade metrics for a candidate fly."""
    curve = candidate_row["curve"]
    prefix = f"{curve}x" if curve != "Spot" else "spot_"

    cols = [prefix + candidate_row["left"], prefix + candidate_row["belly"], prefix + candidate_row["right"]]
    if not all(c in rates_df.columns for c in cols):
        return None

    df3 = rates_df[cols].dropna()
    if weights_df is None or weights_df.dropna().empty:
        return None

    # Compute PCA-weighted fly level as time series
    w = weights_df.dropna()
    common_idx = df3.index.intersection(w.index)
    fly_level = (df3.loc[common_idx].values * w.loc[common_idx].values).sum(axis=1)
    fly_series = pd.Series(fly_level, index=common_idx, name="pca_fly")

    # OU fit on trailing window
    window = config["ou_estimation_window_days"]
    if len(fly_series) < window:
        window = len(fly_series)
    recent = fly_series.iloc[-window:]
    ou = ou_params(recent)
    adf = adf_test(recent)

    return {**ou, **adf, "fly_series": fly_series}

# Run for top candidates
trade_analytics = []
for idx in fly_candidates.head(top_n).index:
    row = fly_candidates.loc[idx]
    w = compute_stage2_weights(rates_df, row, config)
    analytics = compute_trade_analytics(rates_df, row, w, config)
    if analytics is not None:
        trade_analytics.append({
            "fly": f"{row['left']}/{row['belly']}/{row['right']} ({row['curve']})",
            "half_life": analytics["half_life"],
            "investment_horizon": analytics["investment_horizon_875"],
            "theta": analytics["theta"],
            "adf_pvalue": analytics["pvalue"],
            "stationary": analytics["pvalue"] < config["adf_significance"],
            "direction": row["direction"],
        })

analytics_df = pd.DataFrame(trade_analytics)
display(analytics_df)

In [ ]:
# For the top candidate: compare PCA-weighted fly vs 50:50 fly
# Plot scatter of PCA fly vs body yield (should be uncorrelated)
# Plot scatter of PCA fly vs slope (should be uncorrelated)
# Plot scatter of 50:50 fly vs body (should show correlation)

if len(fly_candidates) > 0 and len(trade_analytics) > 0:
    top = fly_candidates.iloc[0]
    curve = top["curve"]
    prefix = f"{curve}x" if curve != "Spot" else "spot_"
    cols = [prefix + top["left"], prefix + top["belly"], prefix + top["right"]]
    df3 = rates_df[cols].dropna()

    w_pca = compute_stage2_weights(rates_df, top, config)
    if w_pca is not None:
        common = df3.index.intersection(w_pca.dropna().index)
        pca_fly = (df3.loc[common].values * w_pca.loc[common].values).sum(axis=1)
        fly_50_50 = 0.5 * df3.loc[common].iloc[:, 0].values + 0.5 * df3.loc[common].iloc[:, 2].values - df3.loc[common].iloc[:, 1].values
        body = df3.loc[common].iloc[:, 1].values
        slope = df3.loc[common].iloc[:, 2].values - df3.loc[common].iloc[:, 0].values

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        axes[0, 0].scatter(body, pca_fly, s=5, alpha=0.5)
        axes[0, 0].set_title(f"PCA Fly vs Body Yield (r={np.corrcoef(body, pca_fly)[0,1]:.3f})")
        axes[0, 0].set_xlabel("Body Yield"); axes[0, 0].set_ylabel("PCA Fly")

        axes[0, 1].scatter(slope, pca_fly, s=5, alpha=0.5)
        axes[0, 1].set_title(f"PCA Fly vs Slope (r={np.corrcoef(slope, pca_fly)[0,1]:.3f})")
        axes[0, 1].set_xlabel("Slope"); axes[0, 1].set_ylabel("PCA Fly")

        axes[1, 0].scatter(body, fly_50_50, s=5, alpha=0.5)
        axes[1, 0].set_title(f"50:50 Fly vs Body Yield (r={np.corrcoef(body, fly_50_50)[0,1]:.3f})")
        axes[1, 0].set_xlabel("Body Yield"); axes[1, 0].set_ylabel("50:50 Fly")

        axes[1, 1].scatter(slope, fly_50_50, s=5, alpha=0.5)
        axes[1, 1].set_title(f"50:50 Fly vs Slope (r={np.corrcoef(slope, fly_50_50)[0,1]:.3f})")
        axes[1, 1].set_xlabel("Slope"); axes[1, 1].set_ylabel("50:50 Fly")

        plt.suptitle(f"Validation: {top['left']}/{top['belly']}/{top['right']} ({top['curve']})")
        plt.tight_layout()
        plt.show()

In [ ]:
# For the spot curve: plot PC1/PC2/PC3 loading shapes
if "Spot" in pca_results:
    spot_result = pca_results["Spot"]
    latest_dt = sorted(spot_result.loadings.keys())[-1]
    ldg = spot_result.loadings[latest_dt]
    ve = spot_result.variance_explained.loc[latest_dt]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    tenors = curve_groups["Spot"]
    x = range(len(tenors))
    for k in range(3):
        axes[k].bar(x, ldg[:, k])
        axes[k].set_xticks(x)
        axes[k].set_xticklabels([t.replace("spot_", "") for t in tenors], rotation=45, fontsize=8)
        axes[k].set_title(f"PC{k+1} ({ve.iloc[k]*100:.1f}% var)")
        axes[k].axhline(y=0, color="black", linewidth=0.5)
    plt.suptitle(f"PCA Loadings — USD SOFR Spot Curve ({latest_dt.strftime('%Y-%m-%d')})")
    plt.tight_layout()
    plt.show()